In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
import pandas as pd
import torch
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

In [ ]:
data_tsv = "./dataset/clean.tsv"
prompts_tsv = "./dataset/prompts.tsv" 

df_data = pd.read_csv(data_tsv, sep='\t')
df_prompts = pd.read_csv(prompts_tsv, sep='\t')

df_merged = pd.merge(df_data, df_prompts, left_on='origin_idx', right_on='number', how='inner')

system_prompt = """You are a scientific visualization expert and prompt engineer.

Create a prompt for an AI image generator that will produce an accurate and visually compelling cover image for a scientific paper.

CRITICAL RULES:
- Scientific accuracy: Visual elements must correctly represent the scientific concept
- No fictional or decorative elements that contradict the research
- Use correct scientific terminology in the prompt
- Output ONLY the prompt, no extra text

PROMPT COMPONENTS:
1. Core scientific concept (what is being shown)
2. Specific visual details (colors, materials, lighting)
3. Style appropriate for the journal/conference
4. Composition and perspective

EXAMPLES:

Article: "Graphene-Based Battery with 10x Capacity"
Prompt: "Cross-section of graphene layered anode material, lithium ions intercalating between graphene sheets, electron flow visualized as golden energy streams, blue and gray color palette, scientific illustration style, cutaway view, detailed material texture"

Article: "Neural Network Explains Visual Cortex Activity"
Prompt: "Artificial neural network architecture overlaid on primate visual cortex diagram, activation patterns shown as colored heatmaps, connections between nodes mimicking biological pathways, academic illustration style, split-view showing both AI and biology, cool blue to warm red gradient
"""

formatted_data = []

for index, row in df_merged.iterrows():
    abstract_text = str(row['abstract'])
    
    teacher_prompt = str(row['prompt']) 
    
    dialogue = {
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Abstract:\n{abstract_text}"},
            {"role": "assistant", "content": teacher_prompt}
        ]
    }
    formatted_data.append(dialogue)

dataset = Dataset.from_list(formatted_data)
print(f"Всего примеров {len(dataset)}")

In [ ]:
max_seq_length = 4096
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

#LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",
)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts}

#Сюда датасет надо
dataset = dataset.map(formatting_prompts_func, batched = True)

print("можно обучать")

In [ ]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to="none"
    ),
)

trainer.train()

In [ ]:
#сохранение модели
model.save_pretrained_gguf(
    "llama_for_article_cover_promt",
    tokenizer,
    quantization_method = "q4_k_m"
)

print("готово")